In [1]:
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import StratifiedKFold, cross_val_score
from sklearn.model_selection import GridSearchCV, RandomizedSearchCV
import joblib

print("Done")

Done


In [2]:
df = pd.read_csv("Dataset.csv")

In [3]:
df.sample(10, ignore_index=True)

,Temperature,Humidity,Moisture,Nitrogen,Phosphorous,Potassium,pH,Rainfall,Soil_Type,Crop_Type,Growth_Stage,Fertilizer Name
0,30,60,53,10,13,22,6.6,160,Clay,Cotton,Seedling,20-20-0
1,36,78,75,1,30,36,8.4,271,Black,Sugarcane,Flowering,14-35-14
2,30,62,52,13,17,22,6.8,153,Sandy,Wheat,Flowering,DAP
3,34,68,65,5,16,33,7.6,225,Black,Sugarcane,Flowering,14-35-14
4,34,71,65,7,16,31,7.3,222,Clay,Cotton,Flowering,14-35-14
5,33,67,64,7,15,28,7.1,192,Loamy,Cotton,Vegetative,14-35-14
6,27,55,44,36,2,0,6.0,111,Loamy,Rice,Seedling,Urea
7,34,68,63,4,21,31,7.7,216,Black,Maize,Flowering,DAP
8,26,50,41,39,0,0,6.1,106,Sandy,Maize,Seedling,Urea
9,23,46,30,44,1,0,6.0,101,Black,Wheat,Seedling,Urea


In [4]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 1825 entries, 0 to 1824
Data columns (total 12 columns):
 #   Column           Non-Null Count  Dtype  
---  ------           --------------  -----  
 0   Temperature      1825 non-null   int64  
 1   Humidity         1825 non-null   int64  
 2   Moisture         1825 non-null   int64  
 3   Nitrogen         1825 non-null   int64  
 4   Phosphorous      1825 non-null   int64  
 5   Potassium        1825 non-null   int64  
 6   pH               1825 non-null   float64
 7   Rainfall         1825 non-null   int64  
 8   Soil_Type        1825 non-null   str    
 9   Crop_Type        1825 non-null   str    
 10  Growth_Stage     1825 non-null   str    
 11  Fertilizer Name  1825 non-null   str    
dtypes: float64(1), int64(7), str(4)
memory usage: 216.4 KB


In [5]:
df.describe()

,Temperature,Humidity,Moisture,Nitrogen,Phosphorous,Potassium,pH,Rainfall
count,1825.000000,1825.000000,1825.000000,1825.000000,1825.000000,1825.000000,1825.000000,1825.000000
mean,28.990685,58.589589,49.569315,21.179178,10.784658,15.815890,6.800000,159.050959
std,5.271512,12.623243,16.496203,17.039274,9.828864,14.449552,0.760266,66.731181
min,18.000000,35.000000,20.000000,0.000000,0.000000,0.000000,5.500000,60.000000
25%,25.000000,48.000000,36.000000,5.000000,2.000000,1.000000,6.200000,103.000000
50%,29.000000,57.000000,47.000000,14.000000,9.000000,16.000000,6.700000,140.000000
75%,33.000000,68.000000,63.000000,39.000000,18.000000,29.000000,7.400000,208.000000
max,40.000000,90.000000,85.000000,50.000000,35.000000,45.000000,8.500000,320.000000


In [ ]:
df["Fertilizer Name"].unique()
# df["Soil_Type"].unique()

<ArrowStringArray>
['Urea', 'Superphosphate', '14-35-14', '20-20-0', 'DAP', '10-26-26']
Length: 6, dtype: str

# Data Preprocessing Pipelines

In [6]:
X = df.drop(columns=["Fertilizer Name"])
y = df["Fertilizer Name"]

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.25, random_state=42, stratify=y)

print("Training rows:", len(X_train))
print("Test rows:", len(X_test))

Training rows: 1368
Test rows: 457


In [9]:
numeric_features = ["Temperature", "Humidity",	"Moisture",	"Nitrogen",	"Phosphorous",	"Potassium",	"pH",	"Rainfall"]

categorical_features = ["Soil_Type",	"Crop_Type",	"Growth_Stage"]

numeric_pipeline = Pipeline([
    ("impute", SimpleImputer(strategy="median")),
    ("scale", StandardScaler())
])

categorical_pipeline = Pipeline([
    ("impute", SimpleImputer(strategy="most_frequent")),
    ("encode", OneHotEncoder(handle_unknown="ignore"))
])

preprocessor = ColumnTransformer([
    ("num", numeric_pipeline, numeric_features),
    ("cat", categorical_pipeline, categorical_features)
])

print("Preprocessor built!")

Preprocessor built!


In [10]:
model_pipeline = Pipeline([
    ("prep", preprocessor),
    ("clf", RandomForestClassifier(random_state=42))
])

model_pipeline.fit(X_train, y_train)

baseline_accuracy = model_pipeline.score(X_test, y_test)

print(f"Baseline test accuracy: {baseline_accuracy:.1%}")

Baseline test accuracy: 96.3%


# Model Evaluation & Cross-Validation

In [11]:
# StratifiedKFold splits the data into 5 folds, keeping the pass/fail ratio balanced in every fold
cv_splitter = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

# cross_val_score automatically: trains on 4 folds, tests on the 5th, and repeats 5 times,
# rotating which fold is held out each time -- returning one score per fold
fold_scores = cross_val_score(model_pipeline, X_train, y_train, cv=cv_splitter)

print("Score on each of the 5 folds:", fold_scores.round(3))

print(f"Mean accuracy: {fold_scores.mean():.1%}")

print(f"Standard deviation: {fold_scores.std():.1%}")

Score on each of the 5 folds: [0.967 0.964 0.967 0.941 0.956]
Mean accuracy: 95.9%
Standard deviation: 1.0%


# Save Machine Learning Model

In [12]:
joblib.dump(model_pipeline, "FERTILIZER-RECOMMENDATION-SYSTEM.joblib")

print("Model saved to FERTILIZER-RECOMMENDATION-SYSTEM.joblib")

Model saved to FERTILIZER-RECOMMENDATION-SYSTEM.joblib


# Testing With Streamlit !

In [ ]:
import streamlit as st

In [ ]:
# load our saved pipeline once when the app starts
model = joblib.load("FERTILIZER-RECOMMENDATION-SYSTEM.joblib")

# a title of the WEB PAGE
st.title("Fertilizer Recommendation System")

# a plain line of instructional text, shown just below the title
st.write("Which Fertilizer should be used ?")

# Streamlit Slider for numeric inputs
hours_studied = st.slider("Hours studied", 0.0, 10.0, 5.0)
attendance_pct = st.slider("Attendance percent", 0.0, 100.0, 75.0)
previous_score = st.slider("Previous test score", 0.0, 100.0, 65.0)

# Streamlit selectbox for categorical input
study_method = st.selectbox("Study method", ["Group", "Solo", "Online"])

if st.button("Predict"):

    # Dataframe that acts as input for predict function
    input_row = pd.DataFrame([{
        "hours_studied": hours_studied,
        "attendance_pct": attendance_pct,
        "previous_score": previous_score,
        "study_method": study_method
    }])

    # the pipeline cleans (impute/scale/encode) AND predicts
    prediction = model.predict(input_row)[0]

    # predict_probability gives us the model's confidence.
    probability = model.predict_proba(input_row)[0][1]

    if prediction == 1:

        st.success(f"Prediction: PASS (confidence: {probability:.0%})")
    else:

        st.error(f"Prediction: FAIL (confidence: {1 - probability:.0%})")